# Activity 5: MLflow, the Run Ledger

**Week 6 Day 4 | One place to record any experiment, whatever produced it**

**Estimated time:** 45 minutes
**Difficulty:** Beginner to Intermediate
**Format:** Individual
**Prerequisites:** [Activity 0](./Activity_0_Environment_and_API_Setup.md) complete, including the `.venv-mlflow` environment

## Read this before you run anything

**This notebook and Activity 6 use a different kernel from the rest of today.** Select `.venv-mlflow`, not the root `.venv`. In VS Code that is **Select Kernel** in the top right, then **Python Environments**, then `.venv-mlflow`.

MLflow requires `pandas` below version 3 and this project is pinned to pandas 3, so MLflow cannot live in the main environment. Activity 0 explains why isolating it is the right call rather than downgrading six weeks of other work.

## The job

MLflow is usually introduced as "the scikit-learn experiment tracker". That description is wrong in a way that matters, because it makes you think you need a different tool for every kind of work.

MLflow is a **run ledger**. You open a run, record what you chose, record what happened, and close it. Whether the thing in the middle was a `RandomForestClassifier`, an OpenAI call, a SQL query, or a shell script is none of MLflow's business.

You are going to prove that by logging two runs that have nothing in common, in exactly the same way, and then comparing them side by side in one table.

## What you will learn

- The four things MLflow stores, and how to decide which one a given number is
- Why a metric is not a parameter, and what you lose by confusing them
- How to record a value that changes during training, not just at the end
- How to find runs again with `search_runs`, including filtering
- How to point the MLflow UI at your runs, which is less obvious than it looks

---
## Setup

First, confirm you are on the right kernel. If this cell raises `ModuleNotFoundError`, you are on the root `.venv` and need to switch to `.venv-mlflow`.

In [ ]:
import os
import mlflow
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("mlflow version:", mlflow.__version__)

## Where your runs are stored, and why you must say so explicitly

MLflow needs a **tracking store**: somewhere to put runs. Left alone it picks a default, and in recent versions that default is a SQLite database file named `mlflow.db` in whatever directory your kernel happens to be running in.

You are going to set it explicitly instead, for two reasons.

The first is ordinary: a default you did not choose is a default you will lose track of.

The second is specific and will cost you an afternoon if you meet it cold. **MLflow builds that default location as a URI, and a URI cannot contain a raw space.** This course lives in folders like `Week 6/Labs/Day 4`. When MLflow encodes that path, the space becomes `%20`, and it then treats the encoded text as a literal folder name, so your runs land in a phantom directory called `Week%206` that you will never think to look in. Your notebook reports success, and the UI shows nothing.

Setting a **relative** URI sidesteps the whole problem, because there is no path in it to encode.

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

print("tracking uri:", mlflow.get_tracking_uri())
print("working dir  :", os.getcwd())

That writes `mlflow.db` into the folder this notebook is running from, whatever that folder is called, spaces or not.

Now create an **experiment**, which is just a named bucket that runs belong to.

In [ ]:
mlflow.set_experiment("week6-day4-tour")

experiment = mlflow.get_experiment_by_name("week6-day4-tour")
print("name        :", experiment.name)
print("experiment id:", experiment.experiment_id)

---
# 1. The four things MLflow stores

Before logging anything real, get the vocabulary straight, because choosing the wrong one is the most common MLflow mistake and it is not reversible without re-running your work.

| Kind | What it is | Changes during a run? | Example |
| :--- | :--- | :--- | :--- |
| **Parameter** | An input you chose | No, fixed for the run | `n_estimators=100`, `temperature=0.0` |
| **Metric** | A number that came out | Yes, can be logged many times | `accuracy=0.94`, `prompt_tokens=310` |
| **Tag** | A label for organising and filtering | Yes | `run_name`, `git_commit`, `owner` |
| **Artifact** | A file the run produced | Written once | a model, a plot, a CSV, the generated text |

The distinction that actually bites is **parameter versus metric**. A parameter is what you *set*; a metric is what you *got*. MLflow stores only the latest value for a parameter but keeps a full history for a metric, so if you log a changing value as a parameter you silently throw away everything except the last one.

Make one throwaway run to see all four in one place.

In [ ]:
with mlflow.start_run(run_name="vocabulary_demo") as run:
    mlflow.log_param("some_input", 42)
    mlflow.log_metric("some_output", 0.87)
    mlflow.set_tag("purpose", "demonstration")
    mlflow.log_text("this is an artifact file", "notes.txt")

    run_id = run.info.run_id

print("run_id:", run_id)

`start_run` is a context manager. The run opens at `with`, and closes when the block ends, which is what marks it `FINISHED`. If an exception escapes the block, MLflow marks the run `FAILED` rather than leaving it open forever, and that is a real reason to prefer the `with` form over calling `start_run`/`end_run` yourself.

Now read the run back and look at what was actually stored.

In [ ]:
stored = mlflow.get_run(run_id)

print("status   :", stored.info.status)
print("params   :", stored.data.params)
print("metrics  :", stored.data.metrics)
print("run name :", stored.data.tags["mlflow.runName"])
print("our tag  :", stored.data.tags["purpose"])

Two details worth noticing.

`params` came back as `{'some_input': '42'}`, a **string**, not the integer you passed. MLflow stores every parameter as text. That is why you cannot do arithmetic on a param after reading it back, and why anything you intend to compare numerically belongs in a metric.

`mlflow.runName` is a tag, not a special field. The run name you pass to `start_run` is stored the same way as the `purpose` tag you set yourself.

---
# 2. A scikit-learn run

Now something real. A small classifier, so there are genuine parameters going in and a genuine metric coming out.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("train:", X_train.shape, "test:", X_test.shape)

In [ ]:
# TODO: Log a RandomForestClassifier run to MLflow.
#
# Inside a `with mlflow.start_run(run_name="rf_baseline"):` block:
#   1. log the params dict below with mlflow.log_params (plural)
#   2. train RandomForestClassifier(**params, random_state=42) on X_train, y_train
#   3. compute accuracy_score(y_test, model.predict(X_test))
#   4. log that accuracy as a METRIC named "accuracy"
#   5. print it
#
# Think about the ordering: log the params BEFORE training, not after. If
# training raises, you still want a record of what you tried. This habit
# matters more as runs get longer.
#
# Hint: mlflow.log_params(params) takes the whole dict at once.
#       mlflow.log_metric("accuracy", accuracy) takes one name and one number.

params = {"n_estimators": 100, "max_depth": 5}

<details>
<summary>Still stuck? Hint 1: the shape of the block</summary>

```python
with mlflow.start_run(run_name="rf_baseline"):
    mlflow.log_params(params)
    model = RandomForestClassifier(**params, random_state=42)
    model.fit(X_train, y_train)
    # accuracy, then log_metric, then print
```

Everything indented under the `with` belongs to that run. Anything you log after the block ends goes nowhere useful.

</details>

<details>
<summary>Still stuck? Hint 2: the accuracy line</summary>

```python
accuracy = accuracy_score(y_test, model.predict(X_test))
mlflow.log_metric("accuracy", accuracy)
```

`log_metric` takes a plain float. If you get a type error, you are probably passing an array, which means you forgot `accuracy_score` and are handing it raw predictions.

</details>

Exactly three lines in that cell were MLflow. The rest is the scikit-learn you already know from Week 6 Day 2. That ratio is the point: tracking is something you add around your work, not a framework you restructure your work into.

## 2.1 Metrics can have a history

A metric is not restricted to one value. Pass a `step` and MLflow records a series, which is how you get the training curves you have seen in MLflow screenshots.

In [ ]:
with mlflow.start_run(run_name="rf_depth_sweep"):
    mlflow.log_param("n_estimators", 100)

    for depth in [1, 2, 3, 5, 8, 13]:
        m = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
        m.fit(X_train, y_train)
        acc = accuracy_score(y_test, m.predict(X_test))
        mlflow.log_metric("accuracy", acc, step=depth)
        print(f"max_depth={depth:>2}  accuracy={acc:.3f}")

One run, six recorded values of the same metric. Open this run in the UI later and `accuracy` is a chart rather than a single number.

Note what was **not** logged as a param: `max_depth`. It changed inside the run, so it is not a fixed input to this run, and logging it as a param would have left only the final value `13` behind with no way to tell which accuracy belonged to which depth.

This is the parameter-versus-metric decision in its sharpest form, and it is worth pausing on: **if a value varies within the run, it cannot be a parameter of that run.** Your alternative design is one run per depth, six runs total, each with `max_depth` as a genuine param. Both are defensible. The sweep-in-one-run version is better for seeing a curve; the six-run version is better for comparing and filtering, and it is what Activity 6 does.

---
# 3. The same ledger, an entirely different kind of work

Here is the claim from the top of the notebook. An OpenAI call has no model object, no training loop, and no scikit-learn anywhere. Log it identically.

In [ ]:
CLAIM_NOTE = (
    "Insured reports rear-end collision at low speed in a parking lot. "
    "Bumper cover cracked, no airbag deployment, other party's insurance "
    "already confirmed liability. Insured requests expedited repair "
    "authorization due to upcoming work travel."
)

print(CLAIM_NOTE)

Before logging it, decide where each piece belongs. `model` and `temperature` are things you chose, so they are params. Token counts came back from the API, so they are metrics. The summary text is a file the run produced, so it is an artifact.

In [ ]:
# TODO: Log an OpenAI call as an MLflow run, using the same four verbs.
#
# Inside `with mlflow.start_run(run_name="claim_summary_llm"):`
#   1. mlflow.log_params(llm_params)
#   2. call the API:
#        client.chat.completions.create(
#            model=llm_params["model"],
#            temperature=llm_params["temperature"],
#            messages=[{"role": "user",
#                       "content": f"Summarize this claim note in one sentence:\n\n{CLAIM_NOTE}"}],
#        )
#   3. log TWO metrics from response.usage:
#        "prompt_tokens"     -> response.usage.prompt_tokens
#        "completion_tokens" -> response.usage.completion_tokens
#   4. save the generated text as an artifact:
#        mlflow.log_text(summary, "summary.txt")
#   5. print the summary
#
# Ask yourself before you write it: why is temperature a param and
# completion_tokens a metric, when both are just numbers? One you chose,
# one you were given. That is the whole rule.

llm_params = {"model": "gpt-4o-mini", "temperature": 0.0}

<details>
<summary>Still stuck? Hint 1: the API call and the text</summary>

```python
response = client.chat.completions.create(
    model=llm_params["model"],
    temperature=llm_params["temperature"],
    messages=[{"role": "user",
               "content": f"Summarize this claim note in one sentence:\n\n{CLAIM_NOTE}"}],
)
summary = response.choices[0].message.content
```

This is the same call shape you used in Activities 2 and 4.

</details>

<details>
<summary>Still stuck? Hint 2: the metrics and the artifact</summary>

```python
mlflow.log_metric("prompt_tokens", response.usage.prompt_tokens)
mlflow.log_metric("completion_tokens", response.usage.completion_tokens)
mlflow.log_text(summary, "summary.txt")
```

`log_text` takes the string first and the filename second, which is the opposite order from what most people guess.

</details>

Read the two TODO cells side by side. `log_params`, `log_metric`, `log_text`, inside a `start_run` block, in both. One trained a forest, one called an API over the network. MLflow could not tell them apart and did not need to.

That is worth more than it first appears. It means the ledger for your classical models and the ledger for your LLM work is **one ledger**, so "which approach was better" is a question you can actually answer instead of comparing a notebook against a spreadsheet.

---
# 4. Getting runs back out

Logging is only half of it. A ledger you cannot query is a diary.

In [ ]:
runs = mlflow.search_runs(experiment_names=["week6-day4-tour"])

print(f"{len(runs)} runs")
print(list(runs.columns)[:12])

`search_runs` returns a pandas `DataFrame`, one row per run, with columns named by convention: `params.*`, `metrics.*`, `tags.*`. Columns exist for the union of everything logged across all runs, so a run that never logged `accuracy` simply has `NaN` there.

That is why the next table has holes in it, and the holes are correct.

In [ ]:
runs[[
    "tags.mlflow.runName",
    "params.n_estimators",
    "params.model",
    "metrics.accuracy",
    "metrics.prompt_tokens",
]]

One table, both kinds of work. The `rf_baseline` row has an accuracy and no token counts; the `claim_summary_llm` row has token counts and no accuracy. Neither is an error, and both are queryable.

## 4.1 Filtering

Pulling every run and filtering in pandas works until you have thousands of runs. `filter_string` pushes the filter into the store instead.

In [ ]:
# TODO: Use search_runs to find only the runs that logged an accuracy
# above 0.85.
#
# search_runs takes a filter_string argument that uses a small SQL-like
# syntax. The pieces you need:
#
#   metrics.accuracy > 0.85          compare a metric
#   params.model = 'gpt-4o-mini'     compare a param, quotes required
#   tags.mlflow.runName = 'rf_baseline'
#
# Combine with `and` if you want more than one condition.
#
# Careful: param comparisons are STRING comparisons, because section 1
# showed params are stored as text. `params.n_estimators > 50` will not do
# what you expect. Metrics are real numbers and compare properly.
#
# Print the run names and accuracies that come back.

good_runs = None

<details>
<summary>Still stuck? Hint: the call</summary>

```python
good_runs = mlflow.search_runs(
    experiment_names=["week6-day4-tour"],
    filter_string="metrics.accuracy > 0.85",
)
print(good_runs[["tags.mlflow.runName", "metrics.accuracy"]])
```

If you get zero rows, check your actual accuracy values in the table above. The threshold may simply be above everything you logged, which is a correct result rather than a bug.

Note that `rf_depth_sweep` logged `accuracy` six times. `search_runs` shows the **last** value it recorded, not the best one. If you want the best, that is a different question and you would log it as its own metric, for example `best_accuracy`.

</details>

---
# 5. The UI

Everything above is also browsable. The command needs one argument, and leaving it off is the single most common way to end up staring at an empty page.

Open a terminal, `cd` into the folder this notebook is running in, activate the MLflow environment, and run:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Then open `http://127.0.0.1:5000`.

**Do not run a bare `mlflow ui`.** Without `--backend-store-uri`, the UI looks in a different place from the one you configured at the top of this notebook, finds nothing, and reports no experiments at all. The store you log to and the store the UI reads must be named the same way, and nothing in either tool will warn you when they disagree.

If you launched it from your `student-work/week6/day4/` folder, that is where your `mlflow.db` is, and the two match.

In the UI, select `week6-day4-tour`, then:

- click `rf_depth_sweep` and open its `accuracy` metric to see the curve from section 2.1
- click `claim_summary_llm` and open **Artifacts** to read the `summary.txt` you logged
- tick two runs and press **Compare** to see params and metrics side by side

Stop the server with `Ctrl+C` when you are done.

---
# Your Turn

Work in your own copy under `student-work/week6/day4/`.

1. **Sweep temperature.** Log three more `claim_summary_llm`-style runs on the same `CLAIM_NOTE`, at `temperature=0.0`, `0.7`, and `1.2`, one run each so temperature is a genuine param. Pull them back with `search_runs` and compare `completion_tokens`. Does temperature change how much the model writes, even though the instruction ("one sentence") never changed? Say whether the difference you see is big enough to be real or could be run-to-run noise.

2. **Add a cost metric.** `gpt-4o-mini` is priced per million tokens (check [the pricing page](https://platform.openai.com/docs/pricing) for current numbers). Add an `estimated_cost_usd` metric to each LLM run. You now have a ledger that answers "what did this experiment cost", which is a question you will be asked.

3. **Log something that is not a number.** Use `mlflow.log_dict` to attach the full `messages` list you sent to the API as a JSON artifact. Then answer this: why is that an artifact rather than a param?

**Stretch goal:** re-run the section 2 classifier three times with `n_estimators` of 10, 100, and 500, as three separate runs. Use `search_runs` with a `filter_string` to find the best one, then explain why `params.n_estimators` sorts in a surprising order if you sort by it. Section 1 already told you the answer.

## What you did

- Set the tracking store explicitly, and learned why a path with a space in it silently breaks MLflow's default.
- Named the four things MLflow stores and used all four in one run.
- Found out that params come back as strings, and that this is why a comparable number belongs in a metric.
- Logged a scikit-learn run and an OpenAI call with the same four verbs, into the same experiment.
- Recorded a metric with a history using `step`, and worked out when that beats one run per setting.
- Queried runs back out as a DataFrame, then pushed a filter into the store instead.
- Pointed the MLflow UI at the right backend, which is the step most people get wrong.

**Next:** [Activity 6](./Activity_6_RAG_Evaluation_LLM_Judge.ipynb) puts this to work: a labelled evaluation set, four retrieval configurations compared fairly, and an LLM acting as judge, all logged here so the comparison is a table instead of an opinion.